In [ ]:
import os
import re
os.environ['OPENAI_API_KEY'] = ''

### Multi-Agent Combination Optimization
1. PC_DIY_System：流水線方式建構一個元件的推薦流程
2. Reducing_Agent：查看清單，選一個商品做降價，持續降價使得總價格低於預算

In [2]:
from agent import Agent
    
class Component_Agent(Agent):
    
    def __init__(self, component_name, model, require, budget):
        
        component_prompt = self.component_prompt_maker(component_name)
        super().__init__(component_name, component_prompt, model)
        
        self.require = require
        self.budget = budget
        
    def forward(self, component_dict):
        
        component_list = self.get_component_list(component_dict)
        
        user_message = {}
        user_message['component_list'] = component_list
        user_message['require'] = self.require
        user_message['budget'] = self.budget
        
        message = self.llm.invoke(user_message)
        self.memory.append(message)
        
        component_dict[self.agent_name] = self.parser_message(message)
        
        return component_dict
        
    def component_prompt_maker(self, component_name):

        prompt = f"""
        你是一個專業的電腦-{component_name}銷售機器人，請根據你的專業，
        根據客戶的需求，提供你覺得客戶可能需要的電腦-{component_name}給他。
        """
        
        prompt = prompt + """
        客戶的預算為：{budget}
        客戶的需求為：{require}
        目前已經有的電腦組合清單為：{component_list}

        請說明推薦原因，並提供產品名稱，範例的回饋訊息如下：
        [ 推薦原因 ] ： 耐用性強，可以使用到最新規格的Intel CPU...
        [ 產品名稱 ] ： Inter 第八代 i9
        [ 產品價格 ] ： 13000
        
        另外有幾點請注意：
        1. 推薦原因的說明請限制在50個字以內
        2. 產品的價格，請回覆一個完整的數字，中間不需要加入逗號
        """

        return prompt
    
    def parser_message(self, message):
        pattern = {
            "reason": r"\[ 推薦原因 \] ：\s*(.*?)\s*\n",
            "name": r"\[ 產品名稱 \] ：\s*(.*?)\s*\n",
            "price": r"\[ 產品價格 \] ：\s*(\d+)"
        }

        results = {key: re.search(regex, message).group(1) for key, regex in pattern.items()}

        return results

class Reducing_Agent(Agent):
    
    def __init__(self, agent_name, prompt, model, require, budget):
        super().__init__(agent_name, prompt, model)
        
        self.require = require
        self.budget = budget
        
        self.max_reducing_limitation = 10

    def forward(self, component_dict):
        
        total_price = self.summary_price(component_dict)
        difference = self.budget - total_price
        accumulated_times = 0
        
        while difference < 0 or accumulated_times >= self.max_reducing_limitation:
            
            component_list = self.get_component_list(component_dict)
            
            user_message = {}
            user_message['component_list'] = component_list
            user_message['total_price'] = total_price
            user_message['difference'] = difference
            user_message['require'] = self.require
            user_message['budget'] = self.budget
            
            message = self.llm.invoke(user_message)
            self.memory.append(message)
        
            replace_component, results = self.parser_replace_message(message)
            component_dict[replace_component] = results
            
            total_price = self.summary_price(component_dict)
            difference = self.budget - total_price # update the difference
            accumulated_times += 1
        
        return component_dict
        
    def parser_replace_message(self, message):
        
        patterns = {
            "replace_component": r"\[ 替換產品 \] ：\s*(.*?)\s*\n",
            "reason": r"\[ 推薦原因 \] ：\s*(.*?)\s*\n",
            "name": r"\[ 產品名稱 \] ：\s*(.*?)\s*\n",
            "price": r"\[ 產品價格 \] ：\s*(\d+)"
        }

        replace_component = re.search(patterns["replace_component"], message).group(1)

        results = {
            "reason": re.search(patterns["reason"], message).group(1),
            "name": re.search(patterns["name"], message).group(1),
            "price": re.search(patterns["price"], message).group(1)
        }

        return replace_component, results
    
    def summary_price(self, component_dict):
    
        return sum([int(component['price']) for component in component_dict.values()])

def get_component_list(component_dict):

    component_list = ''

    for i, (key, value) in enumerate(component_dict.items()):
        
        num = i + 1
        if value:
            component_list += f'{num}. {key}: {value["name"]} | price: {value["price"]}\n'
        else:
            component_list += f'{num}. {key}: None | price: None \n'

    return component_list

def get_budget_from_string(text):
    
    text = text.split('。')[0] # 取出第一句話
    match = re.search(r'\d+', text) # 取數字
    if match:
        budget = int(match.group())
    else:
        print("Not found the number")
    
    return budget

def summary_price(component_dict):

    return sum([int(component['price']) for component in component_dict.values()])

def print_results(component_dict):
    component_list = get_component_list(component_dict)
    print(f'組合清單：\n{component_list}')

    total_price = summary_price(component_dict)
    print(f'總價格：{total_price}')

In [3]:
from multi_agent import Multi_Agent

class PC_DIY_System(Multi_Agent):
    
    def __init__(self, model, require, budget):
        super().__init__()
        
        self.agent_dict = {}
        
        component_name_list = ['Mother board', 'Case', 'CPU', 'GPU', 'Memory', 'Device', 'Power', 'Fan']
        for component_name in component_name_list:
            
            agent = Component_Agent(component_name, model, require, budget)
            self.agent_dict[component_name] = agent
            
        reducing_prompt ="""
            你是一個專業的電腦銷售機器人，目前客戶遇到了一個難題，
            它有一組電腦的清單，清單中含有產品的名稱與價格，
            但清單的總價格超出了預算。
            
            身為專業的電腦銷售機器人，請把它挑出這個電腦清單中，
            哪一個產品可以被替換成更低價格的產品，從而使得整體的價格可以低於或者等於客戶的預算。
            
            客戶的需求為：{require}
            客戶的預算為：{budget}
            目前已經有的電腦組合清單為：{component_list}
            目前已經有的電腦組合清單的總價格：{total_price}
            距離客戶預算的差價：{difference}

            請指出哪一個種類的產品可以被替換，
            推薦的一個更低價格的替代方案。
            請說明推薦原因，並提供產品名稱，範例的回饋訊息如下：
            [ 替換產品 ] ： CPU
            [ 推薦原因 ] ： 耐用性強，可以使用到最新規格的Intel CPU...
            [ 產品名稱 ] ： Inter 第八代 i9
            [ 產品價格 ] ： 13000
            
            另外有幾點請注意：
            1. 推薦原因的說明請限制在50個字以內
            2. 產品的價格，請回覆一個完整的數字，中間不需要加入逗號
            """       
        self.reducing_agent = Reducing_Agent('Reducing_agent', reducing_prompt, model, require, budget)
    
    def execute(self, component_dict):
        
        for agent in self.agent_dict.values():
            
            component_dict = agent(component_dict)
            
        response = self.reducing_agent(component_dict)
        
        return component_dict
    

In [4]:
# Ask for component list
model = 'gpt-4o-mini'

# 金額必須輸入為: xxx元，這句需求的文字陳述後，需要加上。
require = '現在有一個使用者想要50000元的遊戲機器。'
budget = get_budget_from_string(require)

# Build multi-agent system for pc diy 
pc_diy = PC_DIY_System(model, require, budget)

component_dict = {'Mother board': '', 'Case': '', 'CPU': '', 'GPU': '', 
                'Memory': '', 'Device': '', 'Power': '', 'Fan': ''}

component_dict = pc_diy.execute(component_dict)
print_results(component_dict)

組合清單：
1. Mother board: ASUS ROG Strix B550-F Gaming | price: 8590
2. Case: Cooler Master MasterBox Q300L | price: 2990
3. CPU: AMD Ryzen 7 5800X | price: 11500
4. GPU: NVIDIA GeForce GTX 1660 Super | price: 12000
5. Memory: Corsair Vengeance LPX 16GB (2 x 8GB) DDR4 3200MHz | price: 3500
6. Device: Corsair RM750 750W 80 Plus Gold | price: 4000
7. Power: Noctua NH-U12S Redux CPU Cooler | price: 2900
8. Fan: Cooler Master SickleFlow 120 V2 風扇 | price: 750

總價格：46230
